## Load the summary information and preprocessed data

In [1]:
import os
import json
import pandas as pd

from pathlib import Path

from single_sample_gsea import ss_gsea


# Get GEO_ID from folder name (same pattern as prepare_data.ipynb)
GEO_ID = os.path.basename(os.path.dirname(os.path.abspath('__file__')))


description_path = Path('description.json')
preprocessed_csv_path = Path(f"{GEO_ID}_preprocessed.csv")

if not description_path.exists():
    raise FileNotFoundError(
        "description.json not found. Please run 'prepare_data.ipynb' first to generate it."
    )

if not preprocessed_csv_path.exists():
    raise FileNotFoundError(
        f"{preprocessed_csv_path} not found. Please run 'preprocess_data.ipynb' first to generate it."
    )

with open(description_path) as f:
    summary_data = json.load(f)

df = pd.read_csv(preprocessed_csv_path)

print("Data contains", df.shape[0], "samples and", df.shape[1], "columns.")
print("First few rows of the data:")
df.head()


Data contains 95 samples and 5850 columns.
First few rows of the data:


,time to_bcr_(days),biochemical recurrence_within_5_years_of_treatment,sample_title,tissue,treatment,previous treatment,age at_diagnosis,bmi,erg status,gleason grade_group_at_rp,...,COX2,ATP8,ATP6,COX3,ND3,ND4L,ND4,ND5,ND6,CYTB
0,NaN,0.0,3T,Prostate tumor - FFPE,Radical prostatectomy,No,64,26.49,high,G2_Gl7,...,5.653072,3.735675,5.754922,6.377859,4.443601,4.608845,6.780463,6.424128,4.432975,6.144973
1,119.0,1.0,5T,Prostate tumor - FFPE,Radical prostatectomy,No,55,24.62,low,G2_Gl7,...,7.115075,4.565450,6.862079,7.626646,5.357571,5.985145,8.023216,7.360952,5.458961,7.142796
2,1577.0,1.0,6T,Prostate tumor - FFPE,Radical prostatectomy,No,72,23.32,low,G2_Gl7,...,7.177151,4.577825,7.058581,7.599866,5.703526,5.980037,7.990661,7.282877,5.285521,7.433937
3,NaN,0.0,8T,Prostate tumor - FFPE,Radical prostatectomy,No,65,28.65,high,G3_Gl7,...,5.979267,3.950907,6.280321,7.039318,4.867194,4.847216,7.206353,6.636354,4.596222,6.516484
4,NaN,0.0,16T,Prostate tumor - FFPE,Radical prostatectomy,No,59,21.87,high,G4_5_higherGl7,...,6.436773,4.204412,6.245299,6.783016,4.872407,5.230882,7.154715,6.918546,5.172573,6.348771


## Load gene sets

In [2]:
MSIGDB_JSON = Path("../../resources/h.all.v2026.1.Hs.json")

with MSIGDB_JSON.open(encoding="utf-8") as f:
    gene_sets = {name: data["geneSymbols"] for name, data in json.load(f).items()}

print(f"Loaded {len(gene_sets)} gene sets.")


Loaded 50 gene sets.


## Compute single sample GSEA on the preprocessed data

Select only gene columns for the computation.

In [3]:
# IMPORTANT: This is dataset-specific and is the only variable that needs to be changed for each of the dataset.
sample_id_column = 'sample_title'

gene_expression_df = df.iloc[:, summary_data['data_summary']['clinical_features']:].copy()
gene_expression_df.index = df[sample_id_column].astype(str)
gene_expression_df = gene_expression_df.T
gene_expression_df.index.name = "gene"

result = ss_gsea(gene_expression_df, gene_sets)


## Merge the results with the clinical data

The output table should have the same number of rows as before and columns should sum up to clinical data + 50 gene sets.

In [4]:
clinical_df = df.iloc[:, :summary_data["data_summary"]["clinical_features"]].copy()
clinical_df = clinical_df.set_index(sample_id_column)

output_df = pd.concat([clinical_df, result], axis=1)
output_df = output_df.reset_index(drop=False)

output_df.to_csv(f"{GEO_ID}_preprocessed_ssgsea.csv", index=False)

# sanity check (there is 50 gene sets)
assert output_df.shape[1] == summary_data["data_summary"]["clinical_features"] + 50
